# Tâche 1 : Question-réponse avec un modèle génératif pré‑entraîné sans adaptation

L'objectif de cette tâche est d'observer les limites d’un modèle de langage **pré‑entraîné** lorsqu’on lui pose des questions sur des sujets **potentiellement absents** de ses données d’origine.

Dans ce *notebook*, vous faites **uniquement de l’inférence** (c.-à-d. générer des textes) avec un modèle déjà  pré-entraîné. Les paires **questions-réponses** pour évaluer le modèle vous sont fournies (voir fichier *questions.json*). Cette tâche consiste à utiliser les questions comme *prompt*, générer des réponses avec le modèle non modifié, puis évaluer la pertinence des résultats à partir des réponses de référence. Aucun entraînement de modèle n'est effectué pour cette tâche.

> Il est recommandé de faire ce travail pratique en utilisant une carte graphique GPU compatible avec HuggingFace/Pytorch.
> Si votre machine n’en possède pas, vous pouvez utiliser **Google Colab** pour exécuter le *notebook* dans le cloud.

In [5]:
# Vérification de l'environnement
import os
import torch

if torch.cuda.is_available():
    print("GPU disponible :", torch.cuda.get_device_name(0))
else:
    print("Pas de GPU : l'inférence se fera sur CPU (plus lent).")
    print("Pour activer le GPU : Exécution > Modifier le type d'exécution > GPU T4")

print("Fichier trouvé :", os.path.exists("data/test_questions.json"))

GPU disponible : Tesla T4
Fichier trouvé : True


## 1. Chargement du modèle et du tokeniser

In [6]:
model_name = "EleutherAI/pythia-410m"

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device :", device)

Device : cuda


In [8]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
model.eval()
print(f"Nombre de paramètres : {model.num_parameters()/1e6:.0f} M")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  911MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Nombre de paramètres : 405 M


In [9]:
inputs = tokenizer("The capital of Peru is", return_tensors="pt").to(device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=20, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0], skip_special_tokens=True))

The capital of Peru is Lima, the capital of the country.

The capital of Bolivia is Lima,


## 2. Fonctions de test question-réponse  

In [10]:
def build_prompt(question):
    """Construit le prompt à partir d'une question."""
    return f"Question: {question}\nAnswer:"

In [13]:
def generate_answer(question, model, tokenizer, max_new_tokens=100):
    """Génère une réponse et retourne uniquement le texte produit après le prompt."""
    prompt = build_prompt(question)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False, # décodage glouton : résultats reproductibles
            repetition_penalty=1.2, # limite les boucles de répétition
            pad_token_id=tokenizer.eos_token_id,
        )
    # On garde seulement les nouveaux jetons (sans prompt)
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)
    # On coupe si le modèle enchaîne sur une nouvelle question
    answer = answer.split("\nQuestion:")[0].split("\n\n")[0]
    return answer.strip()

In [14]:
q = "What is Mixture of Experts (MoE) and what type of machine learning technique is it?"
print("Q :", q)
print("R :", generate_answer(q, model, tokenizer))

Q : What is Mixture of Experts (MoE) and what type of machine learning technique is it?
R : It's a combination of two techniques. The first one is called "mixture" or "mixing". This means that the model has been trained with different data sets, but they are all from the same source. For example, if you have training set A1 and test set B2 then your model will be able to predict whether an event happened in both cases. If we want to use this method for prediction, we need to split our dataset into train/test pairs so that each pair


## 3. Génération des réponses aux questions avec le modèle

Note : N'oubliez pas d'ajouter un fichier des réponses générées par le modèle dans votre remise du travail.

In [15]:
test_fn = "data/test_questions.json"

In [16]:
import json
from tqdm.auto import tqdm

with open(test_fn, "r", encoding="utf-8") as f:
    test_data = json.load(f)
print(f"{len(test_data)} questions chargées")

results = []
for item in tqdm(test_data):
    generated = generate_answer(item["question"], model, tokenizer)
    results.append({
        "question": item["question"],
        "reference": item["answer"],
        "generated": generated,
    })

# Aperçu des 3 premières réponses
for r in results[:3]:
    print("Q :", r["question"])
    print("R :", r["generated"])
    print("-" * 80)

50 questions chargées


  0%|          | 0/50 [00:00<?, ?it/s]

Q : What is the context window size of GPT-3?
R : The default value for this setting in Windows 7 and later is 1024x768. This means that you can use a larger screen resolution if your computer has more than one monitor connected to it, but not all monitors have the same aspect ratio (e.g., they may be different sizes). You should also check whether there are any other settings related to the display configuration such as refresh rate or color depth.
--------------------------------------------------------------------------------
Q : Explain what is a transformer architecture.
R : A transformer architecture consists of two parts, the input and output layers. The input layer receives an incoming signal from one or more sources (e.g., sensors) that are connected to it through some kind of network topology. In this case, we will call such networks “transformer” networks. Each transformer network has its own set of parameters for each source node in order to transform the received signals i

In [17]:
output_fn = "t1_answers.json"
with open(output_fn, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print("Réponses sauvegardées dans", output_fn)

Réponses sauvegardées dans t1_answers.json


## 4. Évaluation des résultats

Voici du code que vous pouvez utiliser pour évaluer les valeurs de précision, rappel et F1 au niveau de la correspondance des jetons entre 2 séquences de textes. Dans ce cas-ci, on évalue la correspondance entre une réponse de référence (la *bonne* réponse) et la réponse générée par le modèle. Sinon, la librairie *evaluate* de HuggingFace offre des métriques similaires (métrique *squad*).  

In [18]:
import string
import re
from collections import Counter

def remove_articles(text):
    return re.sub(r'\b(a|an|the)\b', ' ', text)

def white_space_fix(text):
    return ' '.join(text.split())

def remove_punc(text):
    exclude = set(string.punctuation)
    return ''.join(ch for ch in text if ch not in exclude)

def lower(text):
    return text.lower()

def normalize_answer(s):
    """Mettre en minuscule et retirer la ponctuation, des déterminants and les espaces."""
    return white_space_fix(remove_articles(remove_punc(lower(s))))

In [20]:
def evaluate_f1(ground_truth, prediction):
    """Normalise les 2 textes, trouve ce qu'il y a en commun et estime précision, rappel et F1."""
    prediction_tokens = normalize_answer(prediction).split()
    ground_truth_tokens = normalize_answer(ground_truth).split()
    common = Counter(prediction_tokens) & Counter(ground_truth_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0, 0.0, 0.0
    precision = 1.0 * num_same / len(prediction_tokens)
    recall = 1.0 * num_same / len(ground_truth_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return precision, recall, f1

In [21]:
import pandas as pd

for r in results:
    p, rc, f1 = evaluate_f1(r["reference"], r["generated"])
    r["precision"], r["recall"], r["f1"] = p, rc, f1

df = pd.DataFrame(results)
pd.set_option("display.max_colwidth", 80)
df[["question", "generated", "precision", "recall", "f1"]]

,question,generated,precision,recall,f1
0,What is the context window size of GPT-3?,The default value for this setting in Windows 7 and later is 1024x768. This ...,0.014925,0.125000,0.026667
1,Explain what is a transformer architecture.,"A transformer architecture consists of two parts, the input and output layer...",0.076923,0.272727,0.120000
2,Define what is a generative pre-trained transformer (GPT).,A GPT is an algorithm that learns to generate new data from the input. It ca...,0.136364,0.400000,0.203390
3,Which sources comprise the weighted pre-training dataset for GPT-3?,The weights of each source are calculated using a combination of three diffe...,0.037500,0.214286,0.063830
4,List the three main classes of language modelling tasks discussed in the con...,The first is to learn a set of transformations that can be applied on an inp...,0.023529,0.222222,0.042553
5,What is knowledge distillation in the context of transformer models?,Knowledge Distillation (KD) is a technique that aims to reduce the number of...,0.214286,0.290323,0.246575
6,What is GPT-4 and who developed it?,The first version of the Generalized Proportional Transformation (GPT) was p...,0.079365,0.294118,0.125000
7,In what year was GPT-4 released in ChatGPT?,"It is not known. The first release date of the program, which was on March 1...",0.047619,0.285714,0.081633
8,Describe what is multimodal capability in the context of GPT-4?,The ability to perform multiple tasks simultaneously.,0.166667,0.038462,0.062500
9,On what date was GPT-4 training completed?,The first day of the month.,0.000000,0.000000,0.000000


In [30]:
print("Moyennes sur", len(df), "questions")
print("="*25)
print(f"Précision : {df['precision'].mean():.3f}")
print(f"Rappel    : {df['recall'].mean():.3f}")
print(f"F1        : {df['f1'].mean():.3f}")

print("\n 3 meilleures réponses (F1)")
print("="*25)
for _, r in df.nlargest(3, "f1").iterrows():
    print(f"[F1={r['f1']:.2f}] Q : {r['question']}")
    print("   Réf :", r["reference"])
    print("   Gén :", r["generated"], "\n")

print("3 pires réponses (F1)")
print("="*25)
for _, r in df.nsmallest(3, "f1").iterrows():
    print(f"[F1={r['f1']:.2f}] Q : {r['question']}")
    print("   Réf :", r["reference"])
    print("   Gén :", r["generated"], "\n")

Moyennes sur 50 questions
Précision : 0.091
Rappel    : 0.205
F1        : 0.107

 3 meilleures réponses (F1)
[F1=0.29] Q : Which companies did Anthropic accuse of knowledge distillation attacks in 2026?
   Réf : Anthropic accused DeepSeek, Moonshot AI, MiniMax Group, and Alibaba Cloud of knowledge distillation attacks and distilling Claude for their models.
   Gén : The company that was accused by the anthropic team, and which is now a part of the anthropic group. 

[F1=0.27] Q : What percentage stake does Microsoft hold in OpenAI Group PBC following the 2025 restructuring?
   Réf : Following the 2025 restructuring, Microsoft holds a 27% stake in OpenAI Group PBC.
   Gén : The company holds a 50% stake. 

[F1=0.25] Q : What does the term 'reinforcement learning' refer to in the context of DeepMind's early work?
   Réf : Reinforcement learning is an algorithm that learns from experience using only raw pixels as data input. DeepMind used this approach to train AI systems to play video ga